# 04 — Gold: Star Schema
**Silver → Gold medallion transformation**

Builds a star schema in **bicycles_gold** optimized for the
`Bicycle RTI Analytics` semantic model (Direct Lake).

### Dimensions
| Table | Grain | Source |
|---|---|---|
| `dim_station` | 1 row / station | silver_station_profile |
| `dim_neighbourhood` | 1 row / neighbourhood | silver_neighbourhood_metrics |
| `dim_time` | 1 row / hour (0–23) | Generated |
| `dim_date` | 1 row / day (365 d) | Generated |
| `dim_weather` | 1 row / unique weather observation hour | silver_weather_hourly |

### Facts
| Table | Grain | Source |
|---|---|---|
| `fact_availability` | 1 row / station / event | silver_availability_events |
| `fact_hourly_demand` | 1 row / neighbourhood / hour | silver_hourly_demand |
| `fact_rebalancing` | 1 row / rebalance candidate | silver_rebalancing_candidates |
| `fact_weather_impact` | 1 row / neighbourhood / hour | silver_weather_bike_joined |

> `forecast_demand` is populated by **NB06 – ML Demand Forecast**, not this notebook.

### Prerequisites
1. Attach **bicycles_gold** as the default lakehouse
2. Add **bicycles_silver** as an additional lakehouse
3. Run **NB03 Silver** and **NB03a Silver Weather** first so all Silver tables exist

In [ ]:
# ============================================================
# CELL 1 — Configuration · Imports · Read Silver
# ============================================================

from pyspark.sql.functions import (
    col, lit, when, coalesce, round as spark_round,
    count, avg as spark_avg, sum as spark_sum,
    min as spark_min, max as spark_max,
    hour, dayofweek, month, year, quarter, weekofyear,
    date_format, to_date, date_trunc, dayofmonth,
    row_number, xxhash64,
    current_timestamp, expr
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, LongType
from pyspark.sql import SparkSession
from datetime import date, timedelta

spark = SparkSession.builder.getOrCreate()

# ── Configuration — use fully qualified table names for pipeline execution ──
# Silver and Gold are NON-SCHEMA lakehouses (no .dbo prefix)
SILVER_LAKEHOUSE = "bicycles_silver"
SILVER_SCHEMA    = f"{SILVER_LAKEHOUSE}"          # No dbo schema

GOLD_LAKEHOUSE   = "bicycles_gold"
GOLD_SCHEMA      = f"{GOLD_LAKEHOUSE}"            # No dbo schema

print(f"Spark version: {spark.version}")
print(f"Processing started at: {date.today()}")
print(f"\nSource: {SILVER_SCHEMA}")
print(f"Target: {GOLD_SCHEMA}")

# ── Verify Silver tables ──────────────────────────────────────
print(f"\nSilver tables ({SILVER_SCHEMA}):")
try:
    spark.sql(f"SHOW TABLES IN {SILVER_SCHEMA}").show(truncate=False)
except Exception as e:
    print(f"  Could not list tables: {e}")

# ── Read Silver tables ────────────────────────────────────────────
def read_silver(table_name):
    return spark.sql(f"SELECT * FROM {SILVER_SCHEMA}.{table_name}")

df_stations      = read_silver("silver_station_profile")
df_neighbourhood = read_silver("silver_neighbourhood_metrics")
df_events        = read_silver("silver_availability_events")
df_hourly        = read_silver("silver_hourly_demand")
df_rebalance     = read_silver("silver_rebalancing_candidates")

# Weather tables (from NB03a Silver Weather)
df_weather_hourly = read_silver("silver_weather_hourly")
df_weather_joined = read_silver("silver_weather_bike_joined")

# Schema check only — skip .count() to avoid scanning 500M+ rows
for _tbl, _df in [
    ("station_profile", df_stations), ("neighbourhood_metrics", df_neighbourhood),
    ("availability_events", df_events), ("hourly_demand", df_hourly),
    ("rebalancing_candidates", df_rebalance), ("weather_hourly", df_weather_hourly),
    ("weather_bike_joined", df_weather_joined),
]:
    print(f"  silver_{_tbl:<30} {len(_df.columns):>3} cols  \u2713")

# ── Incremental Load Utilities ────────────────────────────────
# First run  → Full overwrite (Gold tables don't exist yet)
# Later runs → Incremental for large fact tables (watermark + append)
#              Overwrite for small dimensions (always fresh, stable keys)
#
# Dimension keys use xxhash64 (not row_number) so they are STABLE
# across runs — same natural key always produces the same surrogate key.
# This is essential for incremental fact appends: old fact rows keep
# valid FK references even after dimensions are overwritten.
# ──────────────────────────────────────────────────────────────

def table_exists(fqn):
    """Check if a Delta table already exists."""
    try:
        spark.sql(f"DESCRIBE TABLE {fqn}")
        return True
    except Exception:
        return False

def column_exists(fqn, col_name):
    """Check if a specific column exists in an existing table."""
    if not table_exists(fqn):
        return False
    try:
        schema = spark.sql(f"SELECT * FROM {fqn} LIMIT 0").schema
        return col_name in [f.name for f in schema.fields]
    except Exception:
        return False

def get_max_value(fqn, col_name):
    """Return MAX(col_name), or None if table/column doesn't exist."""
    if not table_exists(fqn):
        return None
    if not column_exists(fqn, col_name):
        return None
    try:
        row = spark.sql(f"SELECT MAX(`{col_name}`) AS wm FROM {fqn}").collect()[0]
        return row["wm"]
    except Exception:
        return None

def get_min_value(fqn, col_name):
    """Return MIN(col_name), or None if table/column doesn't exist."""
    if not table_exists(fqn):
        return None
    if not column_exists(fqn, col_name):
        return None
    try:
        row = spark.sql(f"SELECT MIN(`{col_name}`) AS mv FROM {fqn}").collect()[0]
        return row["mv"]
    except Exception:
        return None

# ── Detect run mode ──────────────────────────────────────────
FIRST_RUN = not table_exists(f"{GOLD_LAKEHOUSE}.fact_availability")
print(f"\nRun mode: {'FIRST RUN (full load)' if FIRST_RUN else 'INCREMENTAL (append new data)'}")


In [ ]:
# ============================================================
# CELL 2 — dim_station (Stable Hash Keys · Deduped)
# ============================================================
# station_key = xxhash64(BikepointID) → deterministic & stable.
# dropDuplicates guarantees exactly one row per station.
# Essential for FK integrity — prevents cartesian explosion in fact joins.

df_dim_station = (
    df_stations
    .dropDuplicates(["BikepointID"])
    .select(
        col("BikepointID").alias("station_id"),
        col("Street").alias("street_address"),
        col("Neighbourhood").alias("neighbourhood"),
        col("Latitude").alias("latitude"),
        col("Longitude").alias("longitude"),
        col("total_docks"),
        col("station_size"),
        col("availability_status").alias("current_status"),
        col("utilization_pct").alias("current_utilization_pct"),
        col("rebalance_priority"),
        when(col("Longitude").between(-0.15, -0.05), "Central")
        .when(col("Longitude") < -0.15, "West")
        .when(col("Longitude") > -0.05, "East")
        .otherwise("Outer").alias("zone"),
    )
    .withColumn("station_key", xxhash64("station_id").cast(LongType()))
    .withColumn("loaded_at", current_timestamp())
    .select(
        "station_key", "station_id", "street_address", "neighbourhood",
        "latitude", "longitude", "total_docks", "station_size",
        "current_status", "current_utilization_pct", "rebalance_priority",
        "zone", "loaded_at",
    )
)

# Safety check
_cnt_check = df_dim_station.count()
print(f"dim_station rows after dedup: {_cnt_check}")
assert _cnt_check < 2000, (
    f"DEDUP FAILED: dim_station has {_cnt_check:,} rows "
    f"(expected ~800). Check Silver station_profile."
)

df_dim_station.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_LAKEHOUSE}.dim_station")

_cnt = spark.sql(
    f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.dim_station"
).collect()[0]["c"]
print(f"✓ dim_station: {_cnt} rows (1 per station, xxhash64 keys)")
df_dim_station.groupBy("zone").count().show()


In [ ]:
# ============================================================
# CELL 3 — dim_neighbourhood (Stable Hash Keys for Incremental)
# ============================================================

df_dim_neighbourhood = (
    df_neighbourhood
    .select(
        col("Neighbourhood").alias("neighbourhood_name"),
        col("station_count"),
        col("total_capacity"),
        col("total_bikes_available"),
        col("total_empty_docks"),
        col("neighbourhood_utilization_pct"),
        col("avg_utilization_pct"),
        col("health_score"),
        col("capacity_status"),
        col("empty_stations"),
        col("full_stations"),
        col("stations_needing_rebalance"),
        col("centroid_lat"),
        col("centroid_lon"),
        when(col("station_count") >= 15, "high_density")
        .when(col("station_count") >= 8, "medium_density")
        .otherwise("low_density").alias("density_tier"),
    )
    .withColumn("neighbourhood_key", xxhash64("neighbourhood_name").cast(LongType()))
    .withColumn("loaded_at", current_timestamp())
    .select(
        "neighbourhood_key", "neighbourhood_name",
        "station_count", "total_capacity",
        "total_bikes_available", "total_empty_docks",
        "neighbourhood_utilization_pct", "avg_utilization_pct",
        "health_score", "capacity_status",
        "empty_stations", "full_stations", "stations_needing_rebalance",
        "centroid_lat", "centroid_lon", "density_tier", "loaded_at",
    )
)

df_dim_neighbourhood.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.dim_neighbourhood")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.dim_neighbourhood").collect()[0]["c"]
print(f"✓ dim_neighbourhood: {_cnt} rows")
display(df_dim_neighbourhood.orderBy("health_score", ascending=False))


In [ ]:
# ============================================================
# CELL 4 — dim_time (24 Hours)
# ============================================================

time_data = []
for h in range(24):
    is_rush = (7 <= h <= 9) or (17 <= h <= 19)
    if 6 <= h <= 9:
        period = "morning_rush"
    elif 10 <= h <= 15:
        period = "midday"
    elif 16 <= h <= 19:
        period = "evening_rush"
    elif 20 <= h <= 23:
        period = "evening"
    else:
        period = "overnight"

    time_data.append((
        h,                                      # time_key
        h,                                      # hour_of_day
        f"{h:02d}:00",                          # hour_label
        period,                                 # time_period
        is_rush,                                # is_rush_hour
        "AM" if h < 12 else "PM",              # am_pm
        "peak" if is_rush else "off_peak",     # demand_tier
    ))

df_dim_time = spark.createDataFrame(time_data, [
    "time_key", "hour_of_day", "hour_label",
    "time_period", "is_rush_hour", "am_pm", "demand_tier",
])

df_dim_time.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.dim_time")

print(f"✓ dim_time: {df_dim_time.count()} rows")
display(df_dim_time)

In [ ]:
# ============================================================
# CELL 5 — dim_date (365 Days)
# ============================================================

start_date = date.today()
date_data = [(start_date + timedelta(days=i),) for i in range(365)]
df_dates = spark.createDataFrame(date_data, ["date_value"])

df_dim_date = df_dates.select(
    (year("date_value") * 10000 + month("date_value") * 100 + dayofmonth("date_value")).alias("date_key"),
    col("date_value"),
    year("date_value").alias("year"),
    quarter("date_value").alias("quarter"),
    month("date_value").alias("month_num"),
    date_format("date_value", "MMMM").alias("month_name"),
    weekofyear("date_value").alias("week_of_year"),
    dayofmonth("date_value").alias("day_of_month"),
    dayofweek("date_value").alias("day_of_week"),
    date_format("date_value", "EEEE").alias("day_name"),
    when(dayofweek("date_value").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    when(month("date_value").isin(3, 4, 5), "spring")
    .when(month("date_value").isin(6, 7, 8), "summer")
    .when(month("date_value").isin(9, 10, 11), "autumn")
    .otherwise("winter").alias("season"),
)

df_dim_date.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.dim_date")

print(f"✓ dim_date: {df_dim_date.count()} rows")
display(df_dim_date.limit(10))

In [ ]:
# ============================================================
# CELL 6 — dim_weather (Stable Hash Keys)
# ============================================================
# One row per unique weather observation hour.
# Uses xxhash64(observation_hour) for stable keys.

df_dim_weather = (
    df_weather_hourly
    .select(
        col("observation_hour"),
        col("observation_date"),
        col("weather_description"),
        col("weather_category"),
        col("weather_severity"),
        col("has_precipitation"),
        col("temperature_c"),
        col("feels_like_c"),
        col("relative_humidity"),
        col("wind_speed_kmh"),
        col("wind_gust_kmh"),
        col("wind_direction_desc"),
        col("visibility_km"),
        col("cloud_cover_pct"),
        col("pressure_mb"),
        col("uv_index"),
        col("is_daytime"),
        col("cycling_comfort_index"),
        col("icon_code"),
        col("weather_latitude"),
        col("weather_longitude"),
        col("location_name"),
    )
    .withColumn("weather_key", xxhash64(col("observation_hour").cast("string")).cast(LongType()))
    .withColumn("time_key", hour("observation_hour").cast(IntegerType()))
    .withColumn("loaded_at", current_timestamp())
    .select(
        "weather_key", "observation_hour", "observation_date", "time_key",
        "weather_description", "weather_category", "weather_severity",
        "has_precipitation", "temperature_c", "feels_like_c",
        "relative_humidity", "wind_speed_kmh", "wind_gust_kmh",
        "wind_direction_desc", "visibility_km", "cloud_cover_pct",
        "pressure_mb", "uv_index", "is_daytime",
        "cycling_comfort_index", "icon_code",
        "weather_latitude", "weather_longitude", "location_name",
        "loaded_at",
    )
)

df_dim_weather.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.dim_weather")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.dim_weather").collect()[0]["c"]
print(f"✓ dim_weather: {_cnt:,} rows")
df_dim_weather.groupBy("weather_severity").count().orderBy("count", ascending=False).show()


In [ ]:
# ============================================================
# CELL 7 — fact_availability (INCREMENTAL)
# ============================================================
# Each silver event becomes a fact row, joined to dim_station for FK.
#
# Incremental strategy:
#   First run       → Full load (overwrite: creates table)
#   Silver overlaps → Silver was overwritten → Gold must also overwrite
#   No overlap      → All Silver events are newer → safe to append
#   No new events   → Skip (no-op)
#
# Overlap detection: if Silver's MIN(event_timestamp) <= Gold's
# MAX(event_timestamp), Silver was overwritten (not appended),
# so Gold must overwrite too to avoid duplicating old events.

_fact_avail_fqn = f"{GOLD_LAKEHOUSE}.fact_availability"

# ── Watermark on source_timestamp (ETL load time set by NB03) ────
# source_timestamp distinguishes old rows from new ones in Silver.
# On incremental Silver runs only NEW rows get a fresh timestamp,
# so Gold filters to just the delta.  If Silver was fully rebuilt
# (all rows share the same source_timestamp), we detect that and
# fall back to a full overwrite so we don't duplicate 561M events.
_gold_has_src_ts = column_exists(_fact_avail_fqn, "source_timestamp")
_silver_has_src_ts = column_exists(
    f"{SILVER_LAKEHOUSE}.silver_availability_events", "source_timestamp"
)

if FIRST_RUN:
    df_events_to_process = df_events
    _load_mode = "OVERWRITE"
    print("  First run → full overwrite")
elif not _gold_has_src_ts or not _silver_has_src_ts:
    # Gold or Silver missing source_timestamp — schema migration
    df_events_to_process = df_events
    _load_mode = "OVERWRITE"
    print("  Schema migration (source_timestamp missing) → full overwrite")
else:
    _gold_max_src = get_max_value(_fact_avail_fqn, "source_timestamp")
    _silver_min_src = get_min_value(
        f"{SILVER_LAKEHOUSE}.silver_availability_events", "source_timestamp"
    )
    if _gold_max_src is None:
        df_events_to_process = df_events
        _load_mode = "OVERWRITE"
        print("  Gold source_timestamp is NULL → full overwrite")
    elif _silver_min_src and _silver_min_src > _gold_max_src:
        # ALL Silver rows newer than Gold's last load → Silver was rebuilt
        df_events_to_process = df_events
        _load_mode = "OVERWRITE"
        print(f"  Silver rebuilt (min_src={_silver_min_src} > gold_max={_gold_max_src})")
        print(f"  → full overwrite to prevent duplicates")
    else:
        # Silver has mix of old + new rows → filter to just new
        df_events_to_process = df_events.filter(
            col("source_timestamp") > lit(_gold_max_src)
        )
        _new_count = df_events_to_process.count()
        if _new_count > 0:
            _load_mode = "APPEND"
            print(f"  Watermark: {_gold_max_src}")
            print(f"  New events to append: {_new_count:,}")
        else:
            _load_mode = "SKIP"
            print(f"  Watermark: {_gold_max_src}")
            print(f"  No new events since last load → SKIP")

# Ensure source_timestamp column exists for watermark tracking
if "source_timestamp" not in df_events_to_process.columns:
    df_events_to_process = df_events_to_process.withColumn(
        "source_timestamp", current_timestamp()
    )

station_lookup = spark.sql(f"SELECT station_key, station_id FROM {GOLD_LAKEHOUSE}.dim_station")

df_fact_availability = (
    df_events_to_process
    .join(station_lookup, df_events_to_process["BikepointID"] == station_lookup["station_id"], "left")
    .withColumn("_hour", hour("event_timestamp"))
    .withColumn(
        "_time_period",
        when(col("_hour").between(6, 9), "morning_rush")
        .when(col("_hour").between(10, 15), "midday")
        .when(col("_hour").between(16, 19), "evening_rush")
        .when(col("_hour").between(20, 23), "evening")
        .otherwise("overnight"),
    )
    .withColumn(
        "_is_rush",
        col("_hour").between(7, 9) | col("_hour").between(17, 19),
    )
    .select(
        xxhash64("event_id").cast(LongType()).alias("availability_key"),
        col("station_key"),
        col("event_id"),
        col("event_timestamp"),
        to_date("event_timestamp").alias("event_date"),
        date_format("event_timestamp", "yyyyMMdd").cast(LongType()).alias("date_key"),
        hour("event_timestamp").alias("time_key"),
        col("No_Bikes").cast(IntegerType()).alias("bikes_available"),
        col("No_Empty_Docks").cast(IntegerType()).alias("empty_docks"),
        col("total_docks").cast(IntegerType()),
        col("utilization_pct").cast(DoubleType()),
        col("event_type"),
        col("is_critical"),
        col("availability_band"),
        col("_is_rush").alias("is_rush_hour"),
        col("_time_period").alias("time_period"),
        col("source_timestamp"),
    )
)

# ── Write based on load mode ─────────────────────────────────
if _load_mode == "OVERWRITE":
    df_fact_availability.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("event_date") \
        .saveAsTable(_fact_avail_fqn)
    _write_msg = "OVERWRITTEN"
elif _load_mode == "APPEND":
    df_fact_availability.write.format("delta").mode("append").saveAsTable(_fact_avail_fqn)
    _write_msg = f"APPENDED {_new_count:,} new rows"
else:
    _write_msg = "SKIPPED (no new data)"

if _load_mode != "SKIP":
    _cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {_fact_avail_fqn}").collect()[0]["c"]
    print(f"✓ fact_availability: {_cnt:,} rows [{_write_msg}]")
else:
    print(f"✓ fact_availability: [{_write_msg}]")

# ── OPTIMIZE only on full overwrite — never on append ─────────
if _load_mode == "OVERWRITE":
    print("  Running OPTIMIZE + ZORDER BY (station_key)...")
    spark.sql(f"OPTIMIZE {_fact_avail_fqn} ZORDER BY (station_key)")
    print("  ✓ OPTIMIZE complete")

In [ ]:
# ============================================================
# CELL 8 — fact_hourly_demand (Aggregated Demand Facts)
# ============================================================

neighbourhood_lookup = spark.sql(f"SELECT neighbourhood_key, neighbourhood_name FROM {GOLD_LAKEHOUSE}.dim_neighbourhood")

df_fact_demand = (
    df_hourly
    .join(
        neighbourhood_lookup,
        df_hourly["Neighbourhood"] == neighbourhood_lookup["neighbourhood_name"],
        "left",
    )
    .select(
        xxhash64("Neighbourhood", "event_hour").cast(LongType()).alias("demand_key"),
        col("neighbourhood_key"),
        col("event_hour"),
        to_date("event_hour").alias("demand_date"),
        date_format("event_hour", "yyyyMMdd").cast(LongType()).alias("date_key"),
        col("hour_of_day").alias("time_key"),
        col("event_count"),
        col("avg_utilization"),
        col("min_bikes"),
        col("max_bikes"),
        col("avg_bikes"),
        col("critical_events"),
        col("rebalance_triggers"),
        col("demand_spike_count"),
        col("active_stations"),
        col("is_rush_hour"),
        when(col("event_count") > 100, "very_high")
        .when(col("event_count") > 50, "high")
        .when(col("event_count") > 20, "medium")
        .otherwise("low").alias("demand_intensity"),
    )
)

df_fact_demand.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.fact_hourly_demand")
_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.fact_hourly_demand").collect()[0]["c"]
print(f"✓ fact_hourly_demand: {_cnt:,} rows")

In [ ]:
# ============================================================
# CELL 9 — fact_rebalancing (Rebalancing Action Facts)
# ============================================================
# Links to dim_station (station_key), dim_neighbourhood (neighbourhood_key),
# dim_time (time_key), and dim_weather (weather_key) for full star schema
# coverage.
#
# Weather FK strategy: dim_weather may only have a few sparse observations.
# Instead of exact hour+date match (which fails when run-time doesn't
# coincide with any observation), we assign the NEAREST weather observation.
# With only a handful of weather rows, this is a tiny broadcast join.

from pyspark.sql.functions import abs as spark_abs, unix_timestamp, broadcast

station_lookup = spark.sql(f"SELECT station_key, station_id FROM {GOLD_LAKEHOUSE}.dim_station")
neighbourhood_lookup_rebal = spark.sql(
    f"SELECT neighbourhood_key, neighbourhood_name FROM {GOLD_LAKEHOUSE}.dim_neighbourhood"
)
df_dim_weather_all = spark.sql(
    f"SELECT weather_key, observation_hour FROM {GOLD_LAKEHOUSE}.dim_weather"
)

df_fact_rebalance = (
    df_rebalance
    # FK 1: station_key
    .join(station_lookup, df_rebalance["BikepointID"] == station_lookup["station_id"], "left")
    # FK 2: neighbourhood_key
    .join(
        neighbourhood_lookup_rebal,
        df_rebalance["Neighbourhood"] == neighbourhood_lookup_rebal["neighbourhood_name"],
        "left",
    )
    .drop("neighbourhood_name")
    .withColumn("assessed_at", current_timestamp())
    # FK 3: time_key
    .withColumn("time_key", hour("assessed_at").cast(IntegerType()))
    .select(
        xxhash64("BikepointID", "Neighbourhood").cast(LongType()).alias("rebalance_key"),
        col("station_key"),
        col("neighbourhood_key"),
        col("time_key"),
        col("BikepointID").alias("station_id"),
        col("Neighbourhood").alias("neighbourhood"),
        col("No_Bikes").cast(IntegerType()).alias("bikes_available"),
        col("No_Empty_Docks").cast(IntegerType()).alias("empty_docks"),
        col("total_docks").cast(IntegerType()),
        col("utilization_pct").cast(DoubleType()),
        col("rebalance_priority").cast(DoubleType()).alias("priority_score"),
        col("recommended_action"),
        col("bikes_to_target").cast(IntegerType()),
        col("availability_status"),
        col("station_size"),
        col("assessed_at"),
        date_format("assessed_at", "yyyyMMdd").cast(LongType()).alias("date_key"),
        (expr("abs(bikes_to_target)") * 5).cast(DoubleType()).alias("estimated_rebalance_cost"),
    )
)

# FK 4: weather_key — NEAREST weather observation
# Because dim_weather is sparse (often < 10 rows), we find the
# observation closest in time to assessed_at.  All rebalancing rows
# share the same assessed_at, so we pick ONE nearest weather_key
# and broadcast it — avoids per-row cross-join entirely.
_weather_count = df_dim_weather_all.count()
if _weather_count > 0:
    # Pick the single nearest weather observation to now
    _nearest = (
        df_dim_weather_all
        .withColumn("_dist", spark_abs(
            unix_timestamp(col("observation_hour")) - unix_timestamp(current_timestamp())
        ))
        .orderBy("_dist")
        .limit(1)
        .select("weather_key")
        .collect()[0]["weather_key"]
    )
    df_fact_rebalance = df_fact_rebalance.withColumn("weather_key", lit(_nearest).cast(LongType()))
    print(f"  Weather FK: assigned nearest weather_key = {_nearest}")
else:
    df_fact_rebalance = df_fact_rebalance.withColumn("weather_key", lit(None).cast(LongType()))
    print("  Weather FK: no weather observations available — weather_key is NULL")

df_fact_rebalance.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.fact_rebalancing")
_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.fact_rebalancing").collect()[0]["c"]
print(f"✓ fact_rebalancing: {_cnt:,} rows")
df_fact_rebalance.groupBy("neighbourhood").agg(
    spark_sum("estimated_rebalance_cost").alias("total_cost"),
    count("*").alias("stations"),
).orderBy("total_cost", ascending=False).show(10, truncate=False)


In [ ]:
# ============================================================
# CELL 10 — fact_weather_impact (Weather Impact on Demand)
# ============================================================
# Joins weather + demand data at the neighbourhood × hour grain.
# Links to dim_weather, dim_neighbourhood, dim_time via surrogate keys.
#
# Weather FK strategy: dim_weather is sparse (few observations).
# For each fact row's event_hour, we find the NEAREST weather
# observation and assign its weather_key.  This is a broadcast
# cross-join (dim_weather is tiny) followed by a window pick.

from pyspark.sql.functions import abs as spark_abs, unix_timestamp, broadcast

# Build lookup tables for FK resolution
df_dim_weather_lookup = spark.sql(
    f"SELECT weather_key, observation_hour FROM {GOLD_LAKEHOUSE}.dim_weather"
)
neighbourhood_lookup = spark.sql(
    f"SELECT neighbourhood_key, neighbourhood_name FROM {GOLD_LAKEHOUSE}.dim_neighbourhood"
)

# ── Nearest-weather FK assignment ─────────────────────────────
# Cross-join each fact row with all dim_weather rows (tiny table),
# compute time distance, keep the closest match per fact row.
_w_nearest = Window.partitionBy("event_hour", "Neighbourhood").orderBy("_time_dist")

df_fact_weather = (
    df_weather_joined
    # Broadcast cross-join with dim_weather (≤ a few dozen rows)
    .crossJoin(broadcast(df_dim_weather_lookup))
    .withColumn("_time_dist", spark_abs(
        unix_timestamp(col("event_hour")) - unix_timestamp(col("observation_hour"))
    ))
    .withColumn("_rn", row_number().over(_w_nearest))
    .filter(col("_rn") == 1)
    .drop("_rn", "_time_dist", "observation_hour")
    # FK: neighbourhood_key
    .join(
        neighbourhood_lookup,
        col("Neighbourhood") == neighbourhood_lookup["neighbourhood_name"],
        "left",
    )
    .drop("neighbourhood_name")
    .withColumn("time_key", hour("event_hour").cast(IntegerType()))
    .select(
        xxhash64("event_hour", "Neighbourhood").cast(LongType()).alias("weather_impact_key"),
        col("weather_key"),
        col("neighbourhood_key"),
        col("time_key"),
        col("event_hour"),
        to_date("event_hour").alias("impact_date"),
        date_format("event_hour", "yyyyMMdd").cast(LongType()).alias("date_key"),
        col("Neighbourhood").alias("neighbourhood"),

        # Demand metrics
        col("event_count"),
        col("weather_adjusted_demand"),
        col("weather_demand_impact"),

        # Weather context (denormalised for analytical convenience)
        col("temperature_c"),
        col("feels_like_c"),
        col("wind_speed_kmh"),
        col("has_precipitation"),
        col("cloud_cover_pct"),
        col("cycling_comfort_index"),
        col("weather_severity"),
        col("weather_category"),
        col("weather_description"),
        col("visibility_km"),

        # Derived analytical columns
        when(col("weather_demand_impact") < 0.5, "severe_suppression")
        .when(col("weather_demand_impact") < 0.7, "moderate_suppression")
        .when(col("weather_demand_impact") < 0.9, "mild_suppression")
        .when(col("weather_demand_impact") <= 1.0, "no_impact")
        .otherwise("boost").alias("impact_category"),

        spark_round(
            col("event_count") - col("weather_adjusted_demand"), 1
        ).alias("demand_gap"),
    )
    .withColumn("loaded_at", current_timestamp())
)

df_fact_weather.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.fact_weather_impact")
_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.fact_weather_impact").collect()[0]["c"]
print(f"✓ fact_weather_impact: {_cnt:,} rows")

# Show impact summary
df_fact_weather.groupBy("impact_category").agg(
    count("*").alias("hours"),
    spark_avg("weather_demand_impact").alias("avg_impact"),
    spark_avg("demand_gap").alias("avg_demand_gap"),
).orderBy("avg_impact").show(truncate=False)


In [ ]:
# ============================================================
# CELL 11 — gold_station_snapshot (Agent-Friendly · ~120 Rows)
# ============================================================
# One row per station with the LATEST event data.
# The ontology / data agent routes "current status" questions here
# instead of scanning 562M rows in fact_availability.
#
# Refreshed every pipeline run — always reflects latest state.

from pyspark.sql.functions import last, first

# Get the most recent event per station
_w_latest = Window.partitionBy("BikepointID").orderBy(col("event_timestamp").desc())

df_snapshot = (
    df_events
    .withColumn("_rn", row_number().over(_w_latest))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# Join with dim_station for full context
station_dim = spark.sql(f"SELECT * FROM {GOLD_LAKEHOUSE}.dim_station")

df_gold_snapshot = (
    df_snapshot
    .join(station_dim, df_snapshot["BikepointID"] == station_dim["station_id"], "inner")
    .select(
        station_dim["station_key"],
        station_dim["station_id"],
        station_dim["street_address"],
        station_dim["neighbourhood"],
        station_dim["latitude"],
        station_dim["longitude"],
        station_dim["total_docks"],
        station_dim["station_size"],
        station_dim["zone"],
        df_snapshot["No_Bikes"].cast(IntegerType()).alias("bikes_available"),
        df_snapshot["No_Empty_Docks"].cast(IntegerType()).alias("empty_docks"),
        df_snapshot["utilization_pct"].cast(DoubleType()),
        df_snapshot["availability_band"],
        df_snapshot["is_critical"],
        col("event_timestamp").alias("last_event_at"),
        col("event_type").alias("last_event_type"),
        col("rebalance_priority"),
        when(col("No_Bikes") == 0, "EMPTY")
        .when(col("No_Empty_Docks") == 0, "FULL")
        .when(col("utilization_pct") >= 0.9, "NEAR_FULL")
        .when(col("utilization_pct") <= 0.1, "NEAR_EMPTY")
        .otherwise("NORMAL").alias("operational_status"),
        when(col("No_Bikes") == 0, True)
        .when(col("No_Empty_Docks") == 0, True)
        .otherwise(False).alias("needs_attention"),
    )
    .withColumn("snapshot_at", current_timestamp())
)

df_gold_snapshot.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.gold_station_snapshot")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.gold_station_snapshot").collect()[0]["c"]
print(f"✓ gold_station_snapshot: {_cnt} rows (1 per station, latest event)")

# Show stations needing attention
df_gold_snapshot.filter(col("needs_attention") == True) \
    .select("station_id", "street_address", "neighbourhood", "operational_status", "bikes_available", "empty_docks") \
    .show(20, truncate=False)


In [ ]:
# ============================================================
# CELL 12 — gold_availability_recent (Last 24h · Agent-Friendly)
# ============================================================
# Hourly aggregation per station for the last 24 hours.
# ~120 stations × 24 hours = ~2,880 rows max.
# The ontology / data agent routes "recent trend" and
# "what happened at station X today?" questions here.
#
# Avoids scanning 562M rows in fact_availability for recent queries.

# Find the max event_timestamp from Silver to define "last 24h"
_max_ts = df_events.select(spark_max("event_timestamp")).collect()[0][0]
_cutoff = _max_ts - expr("INTERVAL 24 HOURS")

# Use Spark SQL interval for the filter
df_recent = df_events.filter(
    col("event_timestamp") >= lit(_max_ts) - expr("INTERVAL 24 HOURS")
)

# Aggregate to hourly grain per station
df_gold_recent = (
    df_recent
    .withColumn("event_hour", date_trunc("hour", col("event_timestamp")))
    .groupBy("BikepointID", "event_hour")
    .agg(
        count("*").alias("event_count"),
        spark_avg("No_Bikes").cast(DoubleType()).alias("avg_bikes"),
        spark_min("No_Bikes").cast(IntegerType()).alias("min_bikes"),
        spark_max("No_Bikes").cast(IntegerType()).alias("max_bikes"),
        spark_avg("No_Empty_Docks").cast(DoubleType()).alias("avg_empty_docks"),
        spark_avg("utilization_pct").cast(DoubleType()).alias("avg_utilization"),
        spark_sum(when(col("is_critical") == True, 1).otherwise(0)).alias("critical_events"),
    )
)

# Join with dim_station for context
station_lookup_recent = spark.sql(
    f"SELECT station_key, station_id, street_address, neighbourhood, zone FROM {GOLD_LAKEHOUSE}.dim_station"
)

df_gold_recent = (
    df_gold_recent
    .join(station_lookup_recent, df_gold_recent["BikepointID"] == station_lookup_recent["station_id"], "left")
    .withColumn("time_key", hour("event_hour").cast(IntegerType()))
    .withColumn("recent_key", xxhash64("BikepointID", col("event_hour").cast("string")).cast(LongType()))
    .select(
        col("recent_key"),
        col("station_key"),
        col("station_id"),
        col("street_address"),
        col("neighbourhood"),
        col("zone"),
        col("event_hour"),
        to_date("event_hour").alias("event_date"),
        date_format("event_hour", "yyyyMMdd").cast(LongType()).alias("date_key"),
        col("time_key"),
        col("event_count"),
        col("avg_bikes"),
        col("min_bikes"),
        col("max_bikes"),
        col("avg_empty_docks"),
        col("avg_utilization"),
        col("critical_events"),
        when(col("avg_utilization") >= 0.9, "critical")
        .when(col("avg_utilization") >= 0.7, "high")
        .when(col("avg_utilization") >= 0.3, "normal")
        .otherwise("low").alias("utilization_band"),
    )
    .withColumn("loaded_at", current_timestamp())
)

df_gold_recent.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{GOLD_LAKEHOUSE}.gold_availability_recent")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.gold_availability_recent").collect()[0]["c"]
print(f"✓ gold_availability_recent: {_cnt:,} rows (hourly per station, last 24h)")
print(f"  Data window: {_max_ts} - 24h")

# Show utilization distribution
df_gold_recent.groupBy("utilization_band").agg(
    count("*").alias("station_hours"),
    spark_avg("avg_utilization").alias("avg_util"),
).orderBy("avg_util", ascending=False).show()


In [ ]:
# ============================================================
# CELL 13 — Gold Layer Summary & Optimization
# ============================================================

print("=" * 70)
print("GOLD STAR SCHEMA COMPLETE")
print("=" * 70)

gold_tables = [
    "dim_station", "dim_neighbourhood", "dim_time", "dim_date", "dim_weather",
    "fact_availability", "fact_hourly_demand", "fact_rebalancing", "fact_weather_impact",
    "gold_station_snapshot", "gold_availability_recent",
]

print(f"\n{'Table':<30} {'Rows':>10} {'Columns':>8}")
print("-" * 52)
for table in gold_tables:
    try:
        _tdf = spark.sql(f"SELECT * FROM {GOLD_LAKEHOUSE}.{table} LIMIT 0")
        _ncols = len(_tdf.columns)
        if table == "fact_availability":
            _nrows = "see above"
        else:
            _nrows = f"{spark.sql(f'SELECT COUNT(*) AS c FROM {GOLD_LAKEHOUSE}.{table}').collect()[0]['c']:>10,}"
        print(f"  {table:<28} {_nrows:>10} {_ncols:>8}")
    except Exception:
        print(f"  {table:<28} {'MISSING':>10} {'---':>8}")

# ── FK integrity checks ──────────────────────────────────────
print("\n" + "=" * 70)
print("REFERENTIAL INTEGRITY CHECKS")
print("=" * 70)

if _load_mode == "OVERWRITE":
    orphan_avail = (
        spark.sql(f"SELECT station_key FROM {GOLD_LAKEHOUSE}.fact_availability")
        .join(spark.sql(f"SELECT station_key FROM {GOLD_LAKEHOUSE}.dim_station"), "station_key", "left_anti")
        .count()
    )
    print(f"  fact_availability    -> dim_station         orphans: {orphan_avail}")
else:
    orphan_avail = 0
    print("  fact_availability    -> dim_station         orphans: \u2014 (skipped, 562M rows)")

orphan_demand = (
    spark.sql(f"SELECT neighbourhood_key FROM {GOLD_LAKEHOUSE}.fact_hourly_demand")
    .join(spark.sql(f"SELECT neighbourhood_key FROM {GOLD_LAKEHOUSE}.dim_neighbourhood"), "neighbourhood_key", "left_anti")
    .count()
)
print(f"  fact_hourly_demand   -> dim_neighbourhood   orphans: {orphan_demand}")

orphan_rebal = (
    spark.sql(f"SELECT station_key, neighbourhood_key, time_key, weather_key FROM {GOLD_LAKEHOUSE}.fact_rebalancing")
    .join(spark.sql(f"SELECT station_key FROM {GOLD_LAKEHOUSE}.dim_station"), "station_key", "left_anti")
    .count()
)
print(f"  fact_rebalancing     -> dim_station         orphans: {orphan_rebal}")

orphan_rebal_hood = (
    spark.sql(f"SELECT station_key, neighbourhood_key, time_key, weather_key FROM {GOLD_LAKEHOUSE}.fact_rebalancing")
    .join(spark.sql(f"SELECT neighbourhood_key FROM {GOLD_LAKEHOUSE}.dim_neighbourhood"), "neighbourhood_key", "left_anti")
    .count()
)
print(f"  fact_rebalancing     -> dim_neighbourhood   orphans: {orphan_rebal_hood}")

orphan_rebal_time = (
    spark.sql(f"SELECT station_key, neighbourhood_key, time_key, weather_key FROM {GOLD_LAKEHOUSE}.fact_rebalancing")
    .join(spark.sql(f"SELECT time_key FROM {GOLD_LAKEHOUSE}.dim_time"), "time_key", "left_anti")
    .count()
)
print(f"  fact_rebalancing     -> dim_time            orphans: {orphan_rebal_time}")

orphan_rebal_weather = (
    spark.sql(f"SELECT station_key, neighbourhood_key, time_key, weather_key FROM {GOLD_LAKEHOUSE}.fact_rebalancing")
    .join(spark.sql(f"SELECT weather_key FROM {GOLD_LAKEHOUSE}.dim_weather"), "weather_key", "left_anti")
    .count()
)
print(f"  fact_rebalancing     -> dim_weather         orphans: {orphan_rebal_weather}")

orphan_wi_weather = (
    spark.sql(f"SELECT weather_key, neighbourhood_key FROM {GOLD_LAKEHOUSE}.fact_weather_impact")
    .join(spark.sql(f"SELECT weather_key FROM {GOLD_LAKEHOUSE}.dim_weather"), "weather_key", "left_anti")
    .count()
)
print(f"  fact_weather_impact  -> dim_weather         orphans: {orphan_wi_weather}")

orphan_wi_hood = (
    spark.sql(f"SELECT weather_key, neighbourhood_key FROM {GOLD_LAKEHOUSE}.fact_weather_impact")
    .join(spark.sql(f"SELECT neighbourhood_key FROM {GOLD_LAKEHOUSE}.dim_neighbourhood"), "neighbourhood_key", "left_anti")
    .count()
)
print(f"  fact_weather_impact  -> dim_neighbourhood   orphans: {orphan_wi_hood}")

total_orphans = orphan_avail + orphan_demand + orphan_rebal + orphan_rebal_hood + orphan_rebal_time + orphan_rebal_weather + orphan_wi_weather + orphan_wi_hood
if total_orphans == 0:
    print("\n  All FK relationships are clean — zero orphan rows")
else:
    print(f"\n  WARNING: {total_orphans} orphan rows detected — review joins above")

print("\n" + "=" * 70)
print("NEXT STEPS:")
print("  0. fact_availability partitioned by event_date + ZORDER by station_key")
print("  1. Run NB06 (ML Demand Forecast) -> builds forecast_demand table (now with weather features)")
print("  2. Open Power BI -> Bicycle RTI Analytics model auto-refreshes via Direct Lake")
print("  3. Build Real-Time Dashboard with KQL queries (NB05)")
print("=" * 70)
